# Data Cleaning

In [1]:
%reset -f

Imports

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import gc

Loading the datasheets

In [3]:
DATA_RAW = Path("../data/raw")

studentInfo = pd.read_csv(DATA_RAW / "studentInfo.csv")
studentVle = pd.read_csv(DATA_RAW / "studentVle.csv")
assessments = pd.read_csv(DATA_RAW / "assessments.csv")
studentAssessment = pd.read_csv(DATA_RAW / "studentAssessment.csv")
courses = pd.read_csv(DATA_RAW / "courses.csv")
vle = pd.read_csv(DATA_RAW / "vle.csv")
studentReg = pd.read_csv(DATA_RAW / "studentRegistration.csv")

### Tables Overview

In [4]:
datasets = {
    "studentInfo": studentInfo,
    "studentVle": studentVle,
    "assessments": assessments,
    "studentAssessment": studentAssessment,
    "courses": courses,
    "vle": vle,
    "studentRegistration": studentReg
}

for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f"{name}")
    print(f"{'='*40}")
    print("Shape:", df.shape)
    print("\nMissing values:\n", df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())


studentInfo
Shape: (32593, 12)

Missing values:
 code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64

Duplicate rows: 0

studentVle
Shape: (10655280, 6)

Missing values:
 code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

Duplicate rows: 787170

assessments
Shape: (206, 6)

Missing values:
 code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

Duplicate rows: 0

studentAssessment
Shape: (173912, 5)

Missing values:
 id_assessment       0
id_student          0
date_submitted      0

Handling Missing Values

In [5]:
studentAssessment['date_submitted'] = (
    studentAssessment['date_submitted']
    .clip(lower=0)
)
studentAssessment['submitted_flag'] = studentAssessment['date_submitted'].notna().astype(int)
studentAssessment['score_missing_flag'] = studentAssessment['score'].isna().astype(int)

Cleaned studentRegistration 
Removed invalid registrations where date\_unregistration exists and is <= 0 \(unregistered before/at course start\)\.
Added derived flags:
registration\_missing\_flag \(if date\_registration is missing\)
unregistered\_flag \(if date\_unregistration exists\)
unregistration\_week \(unregistration date converted to week; missing → \-1\)
Filled missing date\_registration with 0
late\_registration\_flag \(registered after day 0\)

In [6]:
# Keep everyone except those who unregistered before the course started and the date of starting
studentReg = studentReg[~(studentReg['date_unregistration'].notna() & (studentReg['date_unregistration'] <= 0))].copy()

# Features
studentReg['registration_missing_flag'] = studentReg['date_registration'].isna().astype(int)

studentReg['unregistered_flag'] = studentReg['date_unregistration'].notna().astype(int)
studentReg['unregistration_week'] = (studentReg['date_unregistration'] // 7).fillna(-1).astype(int)

studentReg['date_registration'] = studentReg['date_registration'].fillna(0)
studentReg['late_registration_flag'] = (studentReg['date_registration'] > 0).astype(int)

The Courses table and Student Registration table were checked for invalid registration or invalid courses, and the dataset was found to be consistent with no invalid data and no duplicates\.

In [7]:
courses_encoded = courses.copy()
# ensuring numeric values for the presentation length and converting them to no of weeks.
courses_encoded["module_presentation_length"] = pd.to_numeric(courses_encoded["module_presentation_length"],errors="coerce")
courses_encoded["course_weeks"] = np.ceil(courses_encoded["module_presentation_length"] / 7).astype("Int64")

# Validation
print("Unique course week lengths:")
print(courses_encoded["course_weeks"].value_counts(dropna=False))


# Optional validation checks
print("Unique course week lengths:")
print(courses_encoded["course_weeks"].value_counts(dropna=False))

non_stem_modules = {"AAA", "BBB", "GGG"}
# creating course_type_stem column to identify STEM/non-STEM 0=Non-STEM, 1=STEM
courses_encoded["course_type_stem"] = (~courses_encoded["code_module"].astype(str).str.upper().str.strip().isin(non_stem_modules)).astype(int) 

# One-hot encoding done for module AAA, BBB, CCC... and presentations 2013J, 2013B, 2014J...
module_dummies = pd.get_dummies(courses_encoded["code_module"].astype(str).str.upper().str.strip(),prefix="module",drop_first=False)
presentation_dummies = pd.get_dummies(courses_encoded["code_presentation"].astype(str).str.upper().str.strip(),prefix="presentation",drop_first=False)
courses_encoded = pd.concat([courses_encoded, module_dummies, presentation_dummies], axis=1)

# --- Drop original categorical columns AFTER encoding ---
# courses_encoded = courses_encoded.drop(columns=["code_module", "code_presentation"])

display(courses_encoded.head())

del module_dummies
del presentation_dummies
gc.collect()

Unique course week lengths:
course_weeks
39      9
35      8
38      4
34      1
<NA>    0
Name: count, dtype: Int64
Unique course week lengths:
course_weeks
39      9
35      8
38      4
34      1
<NA>    0
Name: count, dtype: Int64


,code_module,code_presentation,module_presentation_length,course_weeks,course_type_stem,module_AAA,module_BBB,module_CCC,module_DDD,module_EEE,module_FFF,module_GGG,presentation_2013B,presentation_2013J,presentation_2014B,presentation_2014J
0,AAA,2013J,268,39,0,True,False,False,False,False,False,False,False,True,False,False
1,AAA,2014J,269,39,0,True,False,False,False,False,False,False,False,False,False,True
2,BBB,2013J,268,39,0,False,True,False,False,False,False,False,False,True,False,False
3,BBB,2014J,262,38,0,False,True,False,False,False,False,False,False,False,False,True
4,BBB,2013B,240,35,0,False,True,False,False,False,False,False,True,False,False,False


33

Demographics cleaning/encoding

In [8]:
studentInfo_clean = studentInfo.copy()

# Pass/Distinction = 0, Fail/Withdrawn = 1
studentInfo_clean["target_at_risk"] = (studentInfo_clean["final_result"].isin(["Fail", "Withdrawn"]).astype(int))

#binary encodings for gender and disability
studentInfo_clean["gender_bin"] = (studentInfo_clean["gender"].astype(str).str.upper() == "M").astype(int)
studentInfo_clean["disability_bin"] = (studentInfo_clean["disability"].astype(str).str.upper() == "Y").astype(int)

# ordinal encoding for age group and missing age flag
age_map = {
    "0-35": 0,
    "35-55": 1,
    "55<=": 2,
    "55+": 2
}
studentInfo_clean["age_band_ord"] = studentInfo_clean["age_band"].map(age_map)
studentInfo_clean["age_band_missing_flag"] = studentInfo_clean["age_band"].isna().astype(int)

# highest education ordinal encoding and missing flag
edu_map = {
    "No Formal quals": 0,
    "Lower Than A Level": 1,
    "A Level or Equivalent": 2,
    "HE Qualification": 3,
    "Post Graduate Qualification": 4
}
studentInfo_clean["highest_education_ord"] = studentInfo_clean["highest_education"].map(edu_map)
studentInfo_clean["highest_education_missing_flag"] = studentInfo_clean["highest_education"].isna().astype(int)

# Region - ohe as per paper
# region_ohe = pd.get_dummies(
#     studentInfo_clean["region"],
#     prefix="region",
#     dummy_na=True
# )
# studentInfo_clean = pd.concat([studentInfo_clean, region_ohe], axis=1)

# nation level ohe as request
def map_region_to_nation(region):
    if pd.isna(region):
        return "Unknown"

    r = str(region).lower()

    if "scotland" in r:
        return "Scotland"
    if "wales" in r:
        return "Wales"
    if "ireland" in r:
        return "Northern_Ireland"
    if "england" in r or "region" in r:
        return "England"

    return "Other"

studentInfo_clean["nation"] = studentInfo_clean["region"].apply(map_region_to_nation)

nation_ohe = pd.get_dummies(studentInfo_clean["nation"],prefix="nation",dummy_na=False)
studentInfo_clean = pd.concat([studentInfo_clean, nation_ohe], axis=1)

# imd-band - categorical not ordinal as paper
studentInfo_clean["imd_band"] = studentInfo_clean["imd_band"].fillna("Unknown")
imd_ohe = pd.get_dummies(studentInfo_clean["imd_band"],prefix="imd",dummy_na=False)
studentInfo_clean = pd.concat([studentInfo_clean, imd_ohe], axis=1)

# drop the original columns for the encoded values
# studentInfo_clean.drop(columns=["gender", "disability", "age_band","highest_education", "region", "nation","final_result","imd_band"],inplace=True)

display(studentInfo_clean.head())


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,imd_10-20,imd_20-30%,imd_30-40%,imd_40-50%,imd_50-60%,imd_60-70%,imd_70-80%,imd_80-90%,imd_90-100%,imd_Unknown
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,False,False,False,False,False,False,False,False,True,False
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,False,True,False,False,False,False,False,False,False,False
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,False,False,True,False,False,False,False,False,False,False
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,False,False,False,False,True,False,False,False,False,False
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,False,False,False,False,True,False,False,False,False,False


In [9]:
print(studentInfo_clean.columns.tolist())

['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'target_at_risk', 'gender_bin', 'disability_bin', 'age_band_ord', 'age_band_missing_flag', 'highest_education_ord', 'highest_education_missing_flag', 'nation', 'nation_England', 'nation_Northern_Ireland', 'nation_Scotland', 'nation_Wales', 'imd_0-10%', 'imd_10-20', 'imd_20-30%', 'imd_30-40%', 'imd_40-50%', 'imd_50-60%', 'imd_60-70%', 'imd_70-80%', 'imd_80-90%', 'imd_90-100%', 'imd_Unknown']


Aggregating the studentVle sum\_click per day and merging the activity\_type to the student\_vle\_daily

In [10]:
# VLE PIPELINE: Raw -> Daily -> Enriched -> Weekly -> delete unused dfs

#Aggregate studentVle to DAILY level
studentVle_daily = (
    studentVle
    .groupby(["id_student", "code_module", "code_presentation", "id_site", "date"], as_index=False)
    .agg(sum_click=("sum_click", "sum"))
)
print("studentVle_daily shape:", studentVle_daily.shape)

#Attach activity_type (lookup)
vle_lookup = vle[["id_site", "activity_type"]].drop_duplicates()

studentVle_daily_enriched = studentVle_daily.merge(
    vle_lookup,
    on="id_site",
    how="left",
    validate="m:1"
)
print("studentVle_daily_enriched shape (before date>=0):", studentVle_daily_enriched.shape)

#Free space
del studentVle_daily
del vle_lookup
del studentVle
gc.collect()

#course starts at day 0
studentVle_daily_enriched = studentVle_daily_enriched[studentVle_daily_enriched["date"] >= 0].copy()

#Week number
studentVle_daily_enriched["week"] = (studentVle_daily_enriched["date"] // 7) + 1

#Map activity_type -> engagement_dim
CONTENT_TYPES = {
    "subpage", "homepage", "oucontent", "resource", "url", "page",
    "folder", "glossary", "htmlactivity", "dualpane", "repeatactivity"
}
ASSESSMENT_TYPES = {"quiz", "externalquiz", "questionnaire"}
SOCIAL_TYPES = {"forumng", "ouwiki", "oucollaborate", "ouelluminate", "dataplus", "sharedsubpage"}

def map_engagement_dim(activity_type):
    if pd.isna(activity_type):
        return "unknown"
    a = str(activity_type).strip().lower()
    if a in CONTENT_TYPES:
        return "content"
    if a in ASSESSMENT_TYPES:
        return "assessment"
    if a in SOCIAL_TYPES:
        return "social"
    return "other"

studentVle_daily_enriched["engagement_dim"] = studentVle_daily_enriched["activity_type"].apply(map_engagement_dim)

#Keep only columns needed for weekly aggregation + quarter features
studentVle_daily_enriched = studentVle_daily_enriched[
    ["id_student", "code_module", "code_presentation", "week", "sum_click", "engagement_dim"]
].copy()

print("studentVle_daily_enriched shape (trimmed):", studentVle_daily_enriched.shape)
display(studentVle_daily_enriched.head())

# 5) Aggregate to WEEKLY (much smaller than daily)
studentVle_weekly = (
    studentVle_daily_enriched
    .groupby(["id_student", "code_module", "code_presentation", "week", "engagement_dim"], as_index=False)
    .agg(sum_click=("sum_click", "sum"))
)
print("studentVle_weekly shape:", studentVle_weekly.shape)

#Free memory
del studentVle_daily_enriched
gc.collect()

studentVle_daily shape: (8459320, 6)
studentVle_daily_enriched shape (before date>=0): (8459320, 7)
studentVle_daily_enriched shape (trimmed): (7859079, 6)


,id_student,code_module,code_presentation,week,sum_click,engagement_dim
0,6516,AAA,2014J,30,6,social
1,6516,AAA,2014J,30,8,social
2,6516,AAA,2014J,32,9,social
3,6516,AAA,2014J,1,38,social
4,6516,AAA,2014J,1,12,social


studentVle_weekly shape: (1085419, 6)


0

In [11]:
# QUARTER NORMALISATION: Weekly -> attach course windows -> Q1..Q4
KEYS = ["id_student", "code_module", "code_presentation"]
COURSE_KEYS = ["code_module", "code_presentation"]

# Quarter cutoffs per course presentation
course_windows = courses_encoded[COURSE_KEYS + ["course_weeks"]].drop_duplicates().copy()
course_windows["course_weeks"] = pd.to_numeric(course_windows["course_weeks"], errors="coerce")

course_windows["q1_end"] = np.ceil(course_windows["course_weeks"] * 0.25).astype(int)
course_windows["q2_end"] = np.ceil(course_windows["course_weeks"] * 0.50).astype(int)
course_windows["q3_end"] = np.ceil(course_windows["course_weeks"] * 0.75).astype(int)
course_windows["q4_end"] = course_windows["course_weeks"].astype(int)

course_windows["q1_end"] = course_windows["q1_end"].clip(lower=1)
course_windows["q2_end"] = course_windows[["q2_end", "course_weeks"]].min(axis=1)
course_windows["q3_end"] = course_windows[["q3_end", "course_weeks"]].min(axis=1)
course_windows["q4_end"] = course_windows["q4_end"].clip(lower=1)

# Attach course windows to WEEKLY VLE records
vle_q = studentVle_weekly.merge(
    course_windows[COURSE_KEYS + ["course_weeks", "q1_end", "q2_end", "q3_end", "q4_end"]],
    on=COURSE_KEYS,
    how="inner",
    validate="m:1"
).copy()

# Keep only valid week range
vle_q = vle_q[(vle_q["week"] >= 1) & (vle_q["week"] <= vle_q["course_weeks"])].copy()

# free space
del course_windows
gc.collect()

# 3) Quarter feature builder (no-leak normalisation inside each window)
def build_quarter_norm_features(vle_df: pd.DataFrame, q_end_col: str, q_label: str) -> pd.DataFrame:
    df = vle_df[vle_df["week"] <= vle_df[q_end_col]]

    dim_keep = {"content", "assessment", "social"}

    student_dim = (
        df[df["engagement_dim"].isin(dim_keep)]
        .groupby(KEYS + ["engagement_dim"], as_index=False)
        .agg(clicks=("sum_click", "sum"))
    )

    course_dim = (
        df[df["engagement_dim"].isin(dim_keep)]
        .groupby(COURSE_KEYS + ["engagement_dim"], as_index=False)
        .agg(course_clicks=("sum_click", "sum"))
    )

    student_dim = student_dim.merge(course_dim, on=COURSE_KEYS + ["engagement_dim"], how="left", validate="m:1")
    student_dim["norm"] = np.where(
        student_dim["course_clicks"] > 0,
        student_dim["clicks"] / student_dim["course_clicks"],
        0.0
    )

    dim_wide = (
        student_dim.pivot_table(index=KEYS, columns="engagement_dim", values="norm", fill_value=0.0)
        .reset_index()
        .rename(columns={
            "content": f"norm_content_{q_label}",
            "assessment": f"norm_assessment_{q_label}",
            "social": f"norm_social_{q_label}",
        })
    )

    student_total = df.groupby(KEYS, as_index=False).agg(total_clicks=("sum_click", "sum"))
    course_total = df.groupby(COURSE_KEYS, as_index=False).agg(course_total_clicks=("sum_click", "sum"))

    total_norm = student_total.merge(course_total, on=COURSE_KEYS, how="left", validate="m:1")
    total_norm[f"norm_total_{q_label}"] = np.where(
        total_norm["course_total_clicks"] > 0,
        total_norm["total_clicks"] / total_norm["course_total_clicks"],
        0.0
    )
    total_norm = total_norm[KEYS + [f"norm_total_{q_label}"]]

    out = total_norm.merge(dim_wide, on=KEYS, how="left")

    for c in [f"norm_content_{q_label}", f"norm_assessment_{q_label}", f"norm_social_{q_label}"]:
        if c not in out.columns:
            out[c] = 0.0
        else:
            out[c] = out[c].fillna(0.0)

    # Free intermediates
    del student_dim, course_dim, dim_wide, student_total, course_total, total_norm
    gc.collect()

    return out

# 4) Build quarter-wise features Q1..Q4
q1 = build_quarter_norm_features(vle_q, "q1_end", "Q1"); gc.collect()
q2 = build_quarter_norm_features(vle_q, "q2_end", "Q2"); gc.collect()
q3 = build_quarter_norm_features(vle_q, "q3_end", "Q3"); gc.collect()
q4 = build_quarter_norm_features(vle_q, "q4_end", "Q4"); gc.collect()

# 5) Combine quarters into one table
quarter_features_norm = (
    q1.merge(q2, on=KEYS, how="outer")
      .merge(q3, on=KEYS, how="outer")
      .merge(q4, on=KEYS, how="outer")
)

# Fill NaNs (registered but non-interacting students will be filled later after left join too)
norm_cols = [c for c in quarter_features_norm.columns if c not in KEYS]
quarter_features_norm[norm_cols] = quarter_features_norm[norm_cols].fillna(0.0)

print("quarter_features_norm shape:", quarter_features_norm.shape)
display(quarter_features_norm.head())

# Free memory: quarter-specific tables + vle_q no longer needed
del q1, q2, q3, q4
del vle_q
gc.collect()


quarter_features_norm shape: (28500, 19)


,id_student,code_module,code_presentation,norm_total_Q1,norm_assessment_Q1,norm_content_Q1,norm_social_Q1,norm_total_Q2,norm_assessment_Q2,norm_content_Q2,norm_social_Q2,norm_total_Q3,norm_assessment_Q3,norm_content_Q3,norm_social_Q3,norm_total_Q4,norm_assessment_Q4,norm_content_Q4,norm_social_Q4
0,6516,AAA,2014J,0.003833,0.000000,0.004167,0.003052,0.004156,0.000000,0.004493,0.003263,0.004759,0.000000,0.005364,0.003086,0.004744,0.000000,0.005478,0.002794
1,8462,DDD,2013J,0.000667,0.000820,0.000893,0.000226,0.000565,0.001430,0.000745,0.000163,0.000418,0.001171,0.000569,0.000113,0.000351,0.000720,0.000483,0.000093
2,8462,DDD,2014J,0.000017,0.000000,0.000020,0.000013,0.000012,0.000000,0.000014,0.000010,0.000009,0.000000,0.000011,0.000007,0.000008,0.000000,0.000009,0.000006
3,11391,AAA,2013J,0.001814,0.000000,0.001996,0.001360,0.001635,0.000000,0.001736,0.001374,0.001272,0.000000,0.001323,0.001135,0.001396,0.000000,0.001497,0.001138
4,23629,BBB,2013B,0.000179,0.000703,0.000158,0.000154,0.000165,0.000489,0.000144,0.000138,0.000131,0.000332,0.000114,0.000112,0.000115,0.000331,0.000100,0.000096


0

In [24]:
# -------- Weekly clicks per dimension (content/assessment/social) --------
dim_keep = {"content", "assessment", "social"} 
weekly_dim_long = ( studentVle_weekly[studentVle_weekly["engagement_dim"].isin(dim_keep)] .groupby(KEYS + ["week", "engagement_dim"], as_index=False) .agg(clicks=("sum_click", "sum")) )
# Pivot
weekly_dim_wide = (
    weekly_dim_long
    .pivot_table(index=KEYS, columns=["week", "engagement_dim"], values="clicks", fill_value=0)
    .reset_index()
)

# --- Robust flattening: detect week by numeric type, otherwise stringify ---
def _col_to_name(col):
    # key columns come as strings
    if isinstance(col, str):
        return col

    # multiindex tuples
    if isinstance(col, tuple):
        parts = list(col)

        # remove empty/None
        parts = [p for p in parts if p is not None and p != ""]

        # try to find a numeric "week" part
        wk = None
        dim = None

        for p in parts:
            if isinstance(p, (int, np.integer, float, np.floating)) and wk is None:
                wk = int(p)

        # engagement_dim is usually a string like 'content'
        # take the last string part as dim (excluding 'clicks' if present)
        str_parts = [str(p) for p in parts if isinstance(p, str)]
        str_parts = [s for s in str_parts if s.lower() not in {"clicks"}]

        if len(str_parts) > 0:
            dim = str_parts[-1]

        # if we found both, format nicely
        if wk is not None and dim is not None:
            return f"wk{wk:02d}_{dim}"

        # fallback: join everything
        return "_".join([str(p) for p in parts])

    # fallback
    return str(col)

weekly_dim_wide.columns = [_col_to_name(c) for c in weekly_dim_wide.columns]

print("weekly_dim_wide shape:", weekly_dim_wide.shape)
display(weekly_dim_wide.head())


weekly_dim_wide shape: (28500, 120)


,id_student,code_module,code_presentation,wk01_assessment,wk01_content,wk01_social,wk02_assessment,wk02_content,wk02_social,wk03_assessment,...,wk36_social,wk37_assessment,wk37_content,wk37_social,wk38_assessment,wk38_content,wk38_social,wk39_assessment,wk39_content,wk39_social
0,6516,AAA,2014J,0.0,169.0,60.0,0.0,20.0,22.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
1,8462,DDD,2013J,2.0,68.0,11.0,0.0,137.0,9.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,8462,DDD,2014J,0.0,0.0,0.0,0.0,7.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,11391,AAA,2013J,0.0,165.0,18.0,0.0,20.0,0.0,0.0,...,5.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,23629,BBB,2013B,0.0,14.0,9.0,0.0,1.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
#  merging them to form master table
info_reg_courses = (
    studentInfo_clean
    .merge(studentReg, on=KEYS, how="inner")
    .merge(courses_encoded, on=COURSE_KEYS, how="inner")
)

# Merge weekly_dim_wide instead of quarter_features_norm
master_table_initial = info_reg_courses.merge(
    weekly_dim_wide,
    on=KEYS,
    how="left"
)

# Fill missing weekly engagement features with 0
weekly_cols = [c for c in weekly_dim_wide.columns if c not in KEYS]
master_table_initial[weekly_cols] = master_table_initial[weekly_cols].fillna(0.0)

# Cleanup
del info_reg_courses, weekly_cols, weekly_dim_wide
gc.collect()

display(master_table_initial.head())

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,wk36_social,wk37_assessment,wk37_content,wk37_social,wk38_assessment,wk38_content,wk38_social,wk39_assessment,wk39_content,wk39_social
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,5.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,12.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [68]:
print(studentReg.shape)
print(studentInfo_clean.shape)
print(master_table_initial.shape)

(29915, 9)
(32593, 35)
(29915, 71)


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=915c48ff-4287-4f39-af89-419738985e00' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>